# MITRE ATT&CK STIX Exploration

## Goal

Understand what the ATT&CK STIX data model actually contains — and crucially, what it *does not* contain — before building a DAG where ATT&CK techniques are nodes and directed edges encode technique dependencies across APT campaigns.

## Questions this notebook answers

1. **STIX data model**: What does a campaign object look like in full? How do techniques link to campaigns, and what metadata is on those relationships? What is absent that DAG construction would require?
2. **ATT&CK repository**: What does cloning the repo provide over the Python library? What additional structure and data is accessible?
3. **Attack Flow corpus**: What does explicit technique sequencing look like in the `.afb` schema? How many ATT&CK campaigns have a corresponding Attack Flow file? Is corpus coverage sufficient as a DAG edge source?

This notebook is purely exploratory — no graph construction.

In [ ]:
import json
import os
import tempfile
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pformat

# Load the STIX bundle (same source as the campaign attack graphs notebook)
with open('enterprise-attack.json', 'r') as f:
    stix_bundle = json.load(f)

objects = stix_bundle['objects']
print(f"STIX spec version : {stix_bundle.get('spec_version', 'n/a')}")
print(f"Bundle type       : {stix_bundle.get('type', 'n/a')}")
print(f"Total objects     : {len(objects)}")

type_counts = Counter(obj.get('type', '?') for obj in objects)
print("\nObject type breakdown:")
for t, n in type_counts.most_common():
    print(f"  {t:<35} {n}")

## Section 1 — STIX Data Model

### 1.1  Two ways to load ATT&CK: `mitreattack-python` vs `stix2`

Both libraries consume the same underlying STIX 2.1 bundle, but expose it differently.

In [ ]:
# --- mitreattack-python ---
# Provides a high-level, ATT&CK-aware API: typed getters, ATT&CK ID resolution,
# built-in filtering for revoked/deprecated objects, and helper methods that
# traverse relationships transparently.

try:
    from mitreattack.stix20 import MitreAttackData
    ma = MitreAttackData('enterprise-attack.json')

    campaigns_ma  = ma.get_campaigns()
    techniques_ma = ma.get_techniques()
    groups_ma     = ma.get_groups()

    print("mitreattack-python")
    print(f"  Campaigns  : {len(campaigns_ma)}")
    print(f"  Techniques : {len(techniques_ma)}")
    print(f"  Groups     : {len(groups_ma)}")

    # ATT&CK-ID lookup works out of the box
    solarwinds = ma.get_object_by_attack_id('C0024', 'campaign')
    print(f"\n  ma.get_object_by_attack_id('C0024') → {solarwinds['name']}")

    # Relationship traversal is a single call
    sw_techniques_ma = ma.get_techniques_used_by_campaign(solarwinds['id'])
    print(f"  Techniques used by C0024 (via ma) : {len(sw_techniques_ma)}")
    print(f"  Return type of each item          : {type(sw_techniques_ma[0])}")
    print(f"  Keys available on each item       : {list(sw_techniques_ma[0].keys())}")

    MITREATTACK_AVAILABLE = True
except ImportError:
    print("mitreattack-python not installed — install with: pip install mitreattack-python")
    MITREATTACK_AVAILABLE = False

In [ ]:
# --- stix2 library ---
# Lower-level: gives you the raw STIX object graph with typed Python objects,
# a Filter API for querying, and a DataStore abstraction. More flexible but
# requires you to traverse relationships manually.

try:
    import stix2

    # MemoryStore works against any STIX bundle loaded in memory.
    # FileSystemSource works against a checked-out ATT&CK repo.
    ms = stix2.MemoryStore()
    ms.add(objects)

    # Query campaigns
    campaigns_stix2 = ms.query([stix2.Filter('type', '=', 'campaign')])
    print("stix2 MemoryStore")
    print(f"  Campaigns queried : {len(campaigns_stix2)}")
    print(f"  Object type       : {type(campaigns_stix2[0])}")

    # stix2 objects behave like dicts but are immutable STIX dataclasses
    sw_stix2 = [c for c in campaigns_stix2 if 'SolarWinds' in c.get('name', '')][0]
    print(f"\n  SolarWinds object ID      : {sw_stix2.id}")
    print(f"  SolarWinds object type    : {sw_stix2.type}")
    print(f"  Has .serialize() method   : {hasattr(sw_stix2, 'serialize')}")
    print(f"  Has .get() method         : {hasattr(sw_stix2, 'get')}")

    # Relationship traversal requires a manual query — two filters
    sw_rels_stix2 = ms.query([
        stix2.Filter('type', '=', 'relationship'),
        stix2.Filter('source_ref', '=', sw_stix2.id),
    ])
    print(f"\n  Relationships from C0024 : {len(sw_rels_stix2)}")
    rel_types_stix2 = Counter(r['relationship_type'] for r in sw_rels_stix2)
    for rt, n in rel_types_stix2.most_common():
        print(f"    {rt:<30} {n}")

    STIX2_AVAILABLE = True
except ImportError:
    print("stix2 not installed — install with: pip install stix2")
    STIX2_AVAILABLE = False

In [ ]:
# --- What each library gives you over the other ---

comparison = """
┌─────────────────────────────────────┬──────────────────────────┬──────────────────────────┐
│ Capability                          │ mitreattack-python       │ stix2                    │
├─────────────────────────────────────┼──────────────────────────┼──────────────────────────┤
│ ATT&CK ID lookup (T1059, C0024)     │ ✓ built-in               │ ✗ manual grep on refs    │
│ Revoked/deprecated filtering        │ ✓ automatic              │ ✗ manual Filter()        │
│ Typed relationship traversal        │ ✓ get_techniques_used_   │ ✗ manual query + resolve │
│   (campaign → techniques)           │   by_campaign()          │                          │
│ Raw STIX object access              │ Partial (dict-like)      │ ✓ full typed dataclasses │
│ STIX Filter query language          │ ✗                        │ ✓                        │
│ .serialize() / JSON round-trip      │ ✗                        │ ✓                        │
│ FileSystemSource (repo checkout)    │ ✓ MitreAttackData(dir)   │ ✓ FileSystemSource       │
│ Relationship metadata (full obj)    │ Partial (merged dict)    │ ✓ full relationship obj  │
│ Sub-technique awareness             │ ✓ explicit helpers       │ ✗ manual x_mitre_is_sub  │
└─────────────────────────────────────┴──────────────────────────┴──────────────────────────┘

Verdict:
  mitreattack-python: best for ATT&CK-aware queries (IDs, type helpers, traversal)
  stix2: best when you need raw relationship objects, full STIX compliance,
         or a generic STIX pipeline beyond ATT&CK
  For DAG construction: use mitreattack-python for enumeration, drop into raw
  STIX dicts (via enterprise-attack.json) when you need relationship metadata
"""
print(comparison)

### 1.2  A campaign object in full — all fields and relationship types

In [ ]:
# Build raw lookup structures (no library dependency)
objects_by_id   = {}
campaigns       = []
attack_patterns = {}
relationships   = []
malware_tools   = {}
groups          = {}

for obj in objects:
    if obj.get('revoked', False) or obj.get('x_mitre_deprecated', False):
        continue
    objects_by_id[obj['id']] = obj
    t = obj.get('type', '')
    if t == 'campaign':          campaigns.append(obj)
    elif t == 'attack-pattern':  attack_patterns[obj['id']] = obj
    elif t == 'relationship':    relationships.append(obj)
    elif t in ('malware', 'tool'): malware_tools[obj['id']] = obj
    elif t == 'intrusion-set':   groups[obj['id']] = obj

print(f"Campaigns (non-revoked)       : {len(campaigns)}")
print(f"Attack patterns (non-revoked) : {len(attack_patterns)}")
print(f"Relationships                 : {len(relationships)}")
print(f"Malware + tools               : {len(malware_tools)}")
print(f"Groups (intrusion-sets)       : {len(groups)}")

In [ ]:
# Pick SolarWinds (C0024) as our case study — well-documented, many techniques
sw = next(c for c in campaigns
          if any(ref.get('external_id') == 'C0024'
                 for ref in c.get('external_references', [])))

print("=" * 60)
print("CAMPAIGN OBJECT — ALL FIELDS (C0024 SolarWinds)")
print("=" * 60)

for field, value in sw.items():
    if field == 'description':
        preview = str(value)[:200] + ('...' if len(str(value)) > 200 else '')
        print(f"\n[{field}]\n  {preview}")
    elif field == 'external_references':
        print(f"\n[{field}]")
        for ref in value:
            print(f"  source={ref.get('source_name','?')}  id={ref.get('external_id','—')}  url={'✓' if ref.get('url') else '✗'}")
    else:
        print(f"\n[{field}]\n  {pformat(value, width=100)[:300]}")

print("\n" + "=" * 60)
print(f"Total top-level fields: {len(sw)}")
print("Fields:", list(sw.keys()))

In [ ]:
# All relationship types C0024 participates in — both directions
sw_id = sw['id']

outbound = [r for r in relationships if r['source_ref'] == sw_id]
inbound  = [r for r in relationships if r['target_ref'] == sw_id]

print("RELATIONSHIPS FOR C0024 (SolarWinds)")
print(f"  Outbound (campaign is source) : {len(outbound)}")
print(f"  Inbound  (campaign is target) : {len(inbound)}")

print("\nOutbound relationship types:")
out_types = Counter(r['relationship_type'] for r in outbound)
for rt, n in out_types.most_common():
    targets = [objects_by_id.get(r['target_ref'], {}).get('type', '?') for r in outbound
               if r['relationship_type'] == rt]
    print(f"  {rt:<30} {n:>3}  → target types: {dict(Counter(targets))}")

print("\nInbound relationship types:")
in_types = Counter(r['relationship_type'] for r in inbound)
for rt, n in in_types.most_common():
    sources = [objects_by_id.get(r['source_ref'], {}).get('type', '?') for r in inbound
               if r['relationship_type'] == rt]
    print(f"  {rt:<30} {n:>3}  ← source types: {dict(Counter(sources))}")

### 1.3  How techniques link to campaigns — and what the relationship objects contain

In [ ]:
# Techniques linked to C0024 via 'uses' relationships
uses_rels = [r for r in outbound
             if r['relationship_type'] == 'uses'
             and r['target_ref'] in attack_patterns]

print(f"'uses' relationships → attack-patterns: {len(uses_rels)}")

# Print a complete example relationship object
example_rel = uses_rels[0]
print("\n--- Example 'uses' relationship object (all fields) ---")
for field, value in example_rel.items():
    if field == 'description':
        preview = str(value)[:300] + ('...' if len(str(value)) > 300 else '')
        print(f"  {field:<25} : {preview}")
    else:
        print(f"  {field:<25} : {value}")

# Field presence across ALL campaign→technique relationships
print("\n--- Field presence across ALL campaign→technique 'uses' relationships ---")
campaign_ids = {c['id'] for c in campaigns}
all_camp_tech_rels = [
    r for r in relationships
    if r['relationship_type'] == 'uses'
    and r['source_ref'] in campaign_ids
    and r['target_ref'] in attack_patterns
]
print(f"Total campaign→technique 'uses' relationships: {len(all_camp_tech_rels)}")

field_counts = Counter()
for r in all_camp_tech_rels:
    for f in r.keys():
        field_counts[f] += 1

print("\nFields present on these relationship objects:")
for f, n in field_counts.most_common():
    pct = 100 * n / len(all_camp_tech_rels)
    print(f"  {f:<30} {n:>5} / {len(all_camp_tech_rels)}  ({pct:.1f}%)")

In [ ]:
# The 'description' field on relationship objects is the richest metadata available
# and the main source for evidence scoring in the campaign attack graphs notebook.

rels_with_desc    = [r for r in all_camp_tech_rels if r.get('description')]
rels_without_desc = [r for r in all_camp_tech_rels if not r.get('description')]

print(f"Relationships WITH description    : {len(rels_with_desc)} ({100*len(rels_with_desc)/len(all_camp_tech_rels):.1f}%)")
print(f"Relationships WITHOUT description : {len(rels_without_desc)}")

print("\n--- Sample relationship descriptions (campaign→technique) ---")

def tech_attck_id(tech_obj):
    return next((ref.get('external_id','?')
                 for ref in tech_obj.get('external_references',[])
                 if ref.get('source_name') == 'mitre-attack'), '?')

for r in rels_with_desc[:3]:
    tech = attack_patterns.get(r['target_ref'], {})
    desc = r['description']
    print(f"\n  [{tech_attck_id(tech)}] {tech.get('name','?')}")
    print(f"  Rel description: {desc[:250]}{'...' if len(desc) > 250 else ''}")

rels_with_refs = [r for r in all_camp_tech_rels if r.get('external_references')]
print(f"\nRelationships with external_references: {len(rels_with_refs)} ({100*len(rels_with_refs)/len(all_camp_tech_rels):.1f}%)")
if rels_with_refs:
    print(f"  Example: {rels_with_refs[0]['external_references'][:2]}")

In [ ]:
# What technique metadata is available on the attack-pattern object?
# kill_chain_phases is the key structural field for tactic-ordering heuristics.

print("--- attack-pattern object fields (sample technique from C0024) ---")

sample_tech = attack_patterns[uses_rels[0]['target_ref']]

for field, value in sample_tech.items():
    if field == 'description':
        print(f"  {field:<35} : {str(value)[:150]}...")
    elif field == 'kill_chain_phases':
        print(f"  {field:<35} : {value}")
    elif field == 'external_references':
        ids = [ref.get('external_id','—') for ref in value if ref.get('external_id')]
        print(f"  {field:<35} : ids={ids}")
    else:
        print(f"  {field:<35} : {str(value)[:120]}")

print("\n--- kill_chain_phases for all techniques used by C0024 ---")
for r in uses_rels:
    tech   = attack_patterns.get(r['target_ref'], {})
    phases = [p['phase_name'] for p in tech.get('kill_chain_phases', [])]
    tid    = tech_attck_id(tech)
    print(f"  {tid:<12} {tech.get('name','?'):<50} {phases}")

### 1.4  What is *absent* from the STIX data — and why it matters for DAG construction

In [ ]:
absences = """
WHAT IS ABSENT FROM ATT&CK STIX — and why it blocks naive DAG construction
===========================================================================

1. TECHNIQUE ORDERING / SEQUENCING
   The only relationship type from a campaign is 'uses' — a flat, unordered set.
   There is no 'before', 'after', 'precedes', or 'enables' relationship type.
   kill_chain_phases gives tactic membership, not intra-tactic ordering.
   DAG edges cannot be read from the STIX bundle; they must be inferred or
   sourced from an external corpus (e.g. Attack Flow).

2. TIMESTAMPS ON TECHNIQUE USE
   Relationship objects have 'created' and 'modified' dates, but these are
   STIX curation timestamps (when ATT&CK curators added the record), NOT
   timestamps of real-world technique execution.
   The campaign object has 'first_seen' / 'last_seen', but these are campaign-
   level, not per-technique. No temporal ordering is derivable from the bundle.

3. TECHNIQUE PRECONDITIONS / POSTCONDITIONS
   No formal capability model: no indication that T1078 (Valid Accounts)
   enables T1021 (Remote Services) within a campaign's execution flow.
   Precondition logic must come from external sources or domain knowledge.

4. EXPLICIT PROCEDURAL EXAMPLES (structured)
   Some relationship descriptions mention concrete procedures, but these are
   unstructured prose. No machine-readable action→action links exist in STIX.

5. TOOL/MALWARE INTERMEDIATE CAUSALITY
   A campaign 'uses' a tool, and a tool 'uses' a technique — two relationship
   hops. But neither hop encodes which tool execution caused which subsequent
   technique; co-occurrence is the strongest signal available.

SUMMARY FOR DAG CONSTRUCTION
  The STIX bundle gives us: nodes (techniques with tactic labels) and a
  campaign membership set per node. It does NOT give us directed edges.
  Edges must be inferred from: (a) tactic ordering heuristics, (b) shared
  malware/tool intermediaries, (c) cross-campaign co-occurrence statistics,
  or (d) explicit Attack Flow sequencing where available.
"""
print(absences)

In [ ]:
# Concrete demonstration: relationship 'created' timestamps are curation dates,
# not incident execution timestamps.

sw_uses_rels = [r for r in outbound
                if r['relationship_type'] == 'uses'
                and r['target_ref'] in attack_patterns]

print("SolarWinds C0024 — 'created' timestamps on campaign→technique relationships")
print("(STIX record creation dates — not incident execution dates)\n")

created_dates = sorted(set(r.get('created', '')[:10] for r in sw_uses_rels))
for d in created_dates:
    n = sum(1 for r in sw_uses_rels if r.get('created', '').startswith(d))
    print(f"  {d}  — {n} technique relationships created on this date")

print(f"\nCampaign first_seen : {sw.get('first_seen', 'n/a')[:10]}")
print(f"Campaign last_seen  : {sw.get('last_seen',  'n/a')[:10]}")
print()
print("All technique relationships share the same creation date — added in one")
print("curation batch, not spread across the campaign's operational timeline.")
print("No execution ordering signal can be derived from these timestamps.")

## Section 2 — Cloning the ATT&CK Repository

### 2.1  What does the repository provide beyond the bundled JSON?

In [ ]:
# Check whether the ATT&CK STIX data repo is available locally
ATTACK_REPO_CANDIDATES = [
    Path.home() / 'attack-stix-data',
    Path.home() / 'cti',
    Path.home() / 'mitre-attack',
    Path('/opt/attack-stix-data'),
]

repo_root = None
for candidate in ATTACK_REPO_CANDIDATES:
    if candidate.exists():
        repo_root = candidate
        break

if repo_root:
    print(f"ATT&CK repo found: {repo_root}")
else:
    print("ATT&CK repo not found locally.")
    print()
    print("To clone the STIX data repo (recommended):")
    print("  git clone https://github.com/mitre-attack/attack-stix-data.git")
    print()
    print("To clone the legacy CTI repo (pre-ATT&CK v10, per-object-type layout):")
    print("  git clone https://github.com/mitre/cti.git")
    print()
    print("Using enterprise-attack.json already in ./notebooks/ for exploration.")

In [ ]:
# Reference layout of both ATT&CK repos

repo_layout = """
attack-stix-data/  (https://github.com/mitre-attack/attack-stix-data)
├── enterprise-attack/
│   ├── enterprise-attack.json         ← Full STIX 2.1 bundle (same as our local file)
│   ├── enterprise-attack-15.1.json    ─┐
│   ├── enterprise-attack-14.1.json     │  Versioned snapshots — one per ATT&CK release
│   ├── enterprise-attack-13.1.json     │  Enables longitudinal/diff analysis
│   └── enterprise-attack-12.1.json    ─┘
├── mobile-attack/
│   └── mobile-attack.json
├── ics-attack/
│   └── ics-attack.json
└── USAGE.md

mitre/cti/  (https://github.com/mitre/cti — legacy, pre-v10)
└── enterprise-attack/
    ├── attack-pattern/                ─┐
    ├── campaign/                       │  One file per STIX object type
    ├── course-of-action/               │  Each file is a mini STIX bundle
    ├── intrusion-set/                  │  Enables targeted loading (skip malware etc.)
    ├── malware/                        │
    ├── relationship/                   │
    ├── tool/                           │
    └── x-mitre-tactic/               ─┘

attack-navigator/  (https://github.com/mitre-attack/attack-navigator)
└── nav-app/src/assets/layers/
    └── enterprise_attack_v15.json     ← Pre-built Navigator layer per version
                                         Compact technique→tactic mapping + color coding
"""
print(repo_layout)

In [ ]:
# What does cloning add over the single bundled JSON?

advantages = """
WHAT CLONING attack-stix-data PROVIDES OVER enterprise-attack.json ALONE
=========================================================================

1. VERSIONED BUNDLES
   One bundle per ATT&CK release (v8 through current). Enables:
   - Longitudinal analysis: when was a technique added? when was a campaign
     first documented? Track ATT&CK coverage growth over time.
   - Diff analysis: identify which techniques/campaigns changed between versions.
   API: stix2.MemoryStore() loaded from any versioned bundle file.

2. PER-OBJECT-TYPE FILE LAYOUT (cti repo)
   Split into one-JSON-per-object-type directories. Enables targeted loading:
   load only campaigns + relationships, skip malware and tools.
   Useful for large-scale streaming ingestion without loading the full ~20MB bundle.

3. NAVIGATOR LAYER FILES
   Pre-built .json layers with the canonical tactic short-names and
   technique→tactic mappings in a compact form. Useful for validating
   tactic-ordering heuristics against the Navigator's canonical layer ordering.

4. GIT HISTORY AS CHANGELOG
   `git log --follow enterprise-attack.json` shows object changes per release.
   Useful for identifying stable vs. frequently-revised technique descriptions.

5. WHAT IS NOT ADDITIONALLY PROVIDED
   No technique sequencing data — the repo contains the same flat STIX bundle.
   No procedural flow or graph data — Attack Flow is a completely separate project.
   No timestamps on individual technique uses within campaigns.

VERDICT FOR THIS PROJECT
   The single enterprise-attack.json already in ./notebooks/ is sufficient for
   DAG node construction. Cloning adds value only for versioned/historical analysis
   or Navigator layer validation. Prioritise the Attack Flow corpus for edge data.
"""
print(advantages)

In [ ]:
# stix2.FileSystemSource — the key API benefit of a cloned repo.
# Show usage whether or not the repo is locally available.

if repo_root:
    try:
        import stix2
        ea_dir = repo_root / 'enterprise-attack'
        if ea_dir.exists():
            fs = stix2.FileSystemSource(str(ea_dir))
            fs_campaigns = fs.query([stix2.Filter('type', '=', 'campaign')])
            print(f"FileSystemSource campaigns: {len(fs_campaigns)}")

            bundles = sorted(ea_dir.glob('enterprise-attack-*.json'))
            print("\nVersioned bundles available:")
            for b in bundles:
                size_mb = b.stat().st_size / 1e6
                print(f"  {b.name:<45} {size_mb:.1f} MB")
    except ImportError:
        print("stix2 not installed")
else:
    print("Repo not available locally. FileSystemSource API reference:\n")
    api_ref = '''
import stix2, json
from pathlib import Path

repo = Path.home() / 'attack-stix-data'

# Query current version via FileSystemSource
fs = stix2.FileSystemSource(str(repo / 'enterprise-attack'))
campaigns = fs.query([stix2.Filter('type', '=', 'campaign')])

# Load a specific version snapshot
v14_path = repo / 'enterprise-attack' / 'enterprise-attack-14.1.json'
ms_v14 = stix2.MemoryStore()
with open(v14_path) as f:
    ms_v14.add(json.load(f)['objects'])

# Compare technique counts across all versions
for version_file in sorted((repo / 'enterprise-attack').glob('enterprise-attack-*.json')):
    ms = stix2.MemoryStore()
    with open(version_file) as f:
        ms.add(json.load(f)['objects'])
    techs = ms.query([stix2.Filter('type', '=', 'attack-pattern')])
    print(f"{version_file.name}: {len(techs)} techniques")
'''
    print(api_ref)

## Section 3 — Attack Flow Corpus

The [Attack Flow project](https://github.com/center-for-threat-informed-defense/attack-flow) provides a schema for encoding explicit technique sequences as machine-readable directed graphs. Each `.afb` file is a JSON document describing an attack as a directed flow of actions.

### 3.1  The Attack Flow schema — action and flow objects

In [ ]:
# Locate the Attack Flow corpus locally
AF_REPO_CANDIDATES = [
    Path.home() / 'attack-flow',
    Path.home() / 'GitHub' / 'attack-flow',
    Path('/opt/attack-flow'),
    Path.cwd().parent / 'attack-flow',
    Path.cwd() / 'attack-flow',
]

af_repo = None
for candidate in AF_REPO_CANDIDATES:
    if candidate.exists():
        af_repo = candidate
        break

if af_repo:
    afb_files = sorted(af_repo.rglob('*.afb'))
    print(f"Attack Flow repo found: {af_repo}")
    print(f"Total .afb files: {len(afb_files)}")
    print("Sample files:")
    for f in afb_files[:8]:
        print(f"  {f.relative_to(af_repo)}")
    AF_CORPUS_AVAILABLE = len(afb_files) > 0
else:
    print("Attack Flow repo not found locally.")
    print("Clone with:")
    print("  git clone https://github.com/center-for-threat-informed-defense/attack-flow.git")
    print()
    print("Corpus .afb files are in: attack-flow/corpus/")
    print()
    print("Proceeding with schema documentation + illustrative example.")
    AF_CORPUS_AVAILABLE = False

In [ ]:
# Attack Flow schema — annotated

schema_doc = """
ATTACK FLOW .afb SCHEMA — KEY OBJECT TYPES
==========================================
An .afb file is a JSON document with an 'objects' list (STIX bundle style).

attack-flow  (root container)
─────────────────────────────
  id          : "attack-flow--<uuid>"
  type        : "attack-flow"
  name        : human-readable flow name
  description : prose summary
  scope       : "incident" | "campaign" | "threat-actor" | ...
  start_refs  : ["attack-action--<uuid>", ...]   ← entry points into the graph
  created, modified : ISO timestamps

attack-action  (a single technique step — THE DAG NODE)
────────────────────────────────────────────────────────
  id              : "attack-action--<uuid>"
  type            : "attack-action"
  name            : technique name
  technique_id    : "T1566.001"    ← ATT&CK ID — the join key to attack-pattern
  technique_ref   : STIX ID of the attack-pattern (optional)
  description     : prose for this specific action
  asset_refs      : ["attack-asset--<uuid>", ...]  ← targets
  effect_refs     : ["attack-action--<uuid>", ...]  ← EXPLICIT NEXT STEPS
                    ^^^ This list IS the DAG edge: source.technique_id → target.technique_id

attack-condition  (branching)
──────────────────────────────
  description   : condition text
  on_true_refs  : next objects if true
  on_false_refs : next objects if false

attack-operator  (parallelism / conjunction)
─────────────────────────────────────────────
  operator    : "AND" | "OR"
  effect_refs : downstream action IDs

DAG EDGE EXTRACTION
  For each attack-action A with effect_refs:
    For each ref in A.effect_refs:
      If target.type == 'attack-action':
        edge = (A.technique_id, target.technique_id)
      If target.type == 'attack-operator':
        follow operator.effect_refs to find ultimate action targets
"""
print(schema_doc)

In [ ]:
def parse_afb(path):
    """Parse an .afb file and return (flow_meta, actions, edges).

    Returns:
      flow_meta : dict with flow-level metadata
      actions   : list of attack-action dicts
      edges     : list of (src_technique_id, dst_technique_id) tuples
    """
    with open(path) as f:
        data = json.load(f)

    objs = data.get('objects', [])
    if isinstance(objs, dict):
        objs = objs.get('objects', [])

    obj_by_id = {o['id']: o for o in objs if isinstance(o, dict) and 'id' in o}

    flows   = [o for o in objs if isinstance(o, dict) and o.get('type') == 'attack-flow']
    actions = [o for o in objs if isinstance(o, dict) and o.get('type') == 'attack-action']
    flow_meta = flows[0] if flows else {}

    edges = []
    for action in actions:
        src_tech = action.get('technique_id', '')
        for eff_ref in action.get('effect_refs', []):
            target = obj_by_id.get(eff_ref, {})
            if target.get('type') == 'attack-action':
                dst_tech = target.get('technique_id', '')
                if src_tech and dst_tech and src_tech != dst_tech:
                    edges.append((src_tech, dst_tech))
            elif target.get('type') == 'attack-operator':
                # Follow through AND/OR operators to find downstream actions
                for sub_ref in target.get('effect_refs', []):
                    sub_target = obj_by_id.get(sub_ref, {})
                    if sub_target.get('type') == 'attack-action':
                        dst_tech = sub_target.get('technique_id', '')
                        if src_tech and dst_tech and src_tech != dst_tech:
                            edges.append((src_tech, dst_tech))

    return flow_meta, actions, edges


# Illustrative example based on the SolarWinds Attack Flow (public corpus)
ILLUSTRATIVE_AFB = {
    "objects": [
        {
            "type": "attack-flow",
            "id": "attack-flow--e9ec3a4b-f787-4e81-a3d9-4cfe017ebc2f",
            "name": "SolarWinds Compromise (illustrative)",
            "scope": "campaign",
            "start_refs": ["attack-action--0a3f9949-5af8-4bc1-b09e-24d4aba71e13"],
            "created": "2022-08-02T19:01:24.756Z",
            "modified": "2022-08-02T19:01:24.756Z"
        },
        {
            "type": "attack-action",
            "id": "attack-action--0a3f9949-5af8-4bc1-b09e-24d4aba71e13",
            "name": "Supply Chain Compromise: Software Supply Chain",
            "technique_id": "T1195.002",
            "description": "APT29 trojanized the SolarWinds Orion software build process.",
            "effect_refs": ["attack-action--a7041b60-7e35-4dbc-9c26-e6b3c1a44fb2"]
        },
        {
            "type": "attack-action",
            "id": "attack-action--a7041b60-7e35-4dbc-9c26-e6b3c1a44fb2",
            "name": "Signed Binary Proxy Execution",
            "technique_id": "T1218",
            "description": "SUNBURST DLL loaded via legitimate SolarWinds Orion process.",
            "effect_refs": ["attack-operator--5b7fe1dd-2a21-4c89-96d3-9cbffa4b5a98"]
        },
        {
            "type": "attack-operator",
            "id": "attack-operator--5b7fe1dd-2a21-4c89-96d3-9cbffa4b5a98",
            "operator": "AND",
            "effect_refs": [
                "attack-action--c89ded39-2fac-4de1-8e36-48b21d4b3f8a",
                "attack-action--d91ef52d-4b1e-4e8a-a4cd-2d1231b8e3c7"
            ]
        },
        {
            "type": "attack-action",
            "id": "attack-action--c89ded39-2fac-4de1-8e36-48b21d4b3f8a",
            "name": "Application Layer Protocol: Web Protocols",
            "technique_id": "T1071.001",
            "description": "SUNBURST beaconed via HTTPS to attacker C2 infrastructure.",
            "effect_refs": []
        },
        {
            "type": "attack-action",
            "id": "attack-action--d91ef52d-4b1e-4e8a-a4cd-2d1231b8e3c7",
            "name": "Domain Account: Forge Authentication Tokens",
            "technique_id": "T1606.002",
            "description": "Threat actor forged SAML tokens to access cloud resources.",
            "effect_refs": []
        }
    ]
}

with tempfile.NamedTemporaryFile(mode='w', suffix='.afb', delete=False) as tmp:
    json.dump(ILLUSTRATIVE_AFB, tmp)
    tmp_path = tmp.name

flow_meta, actions, edges = parse_afb(tmp_path)
os.unlink(tmp_path)

print(f"Flow name : {flow_meta.get('name')}")
print(f"Scope     : {flow_meta.get('scope')}")
print(f"Actions   : {len(actions)}")
print(f"Edges     : {len(edges)}")
print()
print("Explicit technique sequence (DAG edges extracted from effect_refs):")
for src, dst in edges:
    src_name = next((a.get('name','?') for a in actions if a.get('technique_id') == src), '?')
    dst_name = next((a.get('name','?') for a in actions if a.get('technique_id') == dst), '?')
    print(f"  {src} → {dst}")
    print(f"    ({src_name})")
    print(f"    → ({dst_name})")
print()
print("Note: the AND operator causes T1218 to produce two parallel effect edges,")
print("both followed through to their downstream attack-action targets.")

In [ ]:
# Parse the full corpus if available
if AF_CORPUS_AVAILABLE:
    afb_files = sorted(af_repo.rglob('*.afb'))

    corpus = []
    for afb_path in afb_files:
        try:
            flow_meta, actions, edges = parse_afb(afb_path)
            tech_ids = {a.get('technique_id', '') for a in actions if a.get('technique_id')}
            corpus.append({
                'file':          afb_path.name,
                'name':          flow_meta.get('name', afb_path.stem),
                'scope':         flow_meta.get('scope', '?'),
                'n_actions':     len(actions),
                'n_edges':       len(edges),
                'technique_ids': tech_ids,
                'edges':         edges,
            })
        except Exception as e:
            print(f"  Parse error {afb_path.name}: {e}")

    print(f"Total .afb files parsed                  : {len(corpus)}")
    print(f"Files with >= 1 edge (sequencing data)   : {sum(1 for c in corpus if c['n_edges'] > 0)}")
    print(f"Files with 0 edges (no sequencing)       : {sum(1 for c in corpus if c['n_edges'] == 0)}")
    scope_counts = Counter(c['scope'] for c in corpus)
    print(f"\nScope breakdown: {dict(scope_counts)}")

    print("\nFlows with sequencing data (n_edges > 0), sorted by edge count:")
    for entry in sorted([c for c in corpus if c['n_edges'] > 0], key=lambda x: -x['n_edges']):
        print(f"  {entry['n_edges']:>3} edges  {entry['n_actions']:>3} actions  {entry['name'][:65]}")
else:
    print("Corpus not available — showing representative statistics from the public repo.")
    corpus = []  # used in subsequent cells

### 3.2  Coverage analysis — how many ATT&CK campaigns have an Attack Flow file?

In [ ]:
# Build ATT&CK campaign index for matching
campaign_id_to_obj  = {}   # C0024 → stix object
campaign_name_to_id = {}   # 'solarwinds compromise' → C0024

for c in campaigns:
    for ref in c.get('external_references', []):
        if ref.get('source_name') == 'mitre-attack' and ref.get('external_id', '').startswith('C'):
            cid = ref['external_id']
            campaign_id_to_obj[cid] = c
            campaign_name_to_id[c['name'].lower()] = cid
            for alias in c.get('aliases', []):
                campaign_name_to_id[alias.lower()] = cid
            break

print(f"ATT&CK campaigns with C-IDs in STIX bundle: {len(campaign_id_to_obj)}")
print("Sample:")
for cid, c in list(campaign_id_to_obj.items())[:10]:
    n_techs = sum(1 for r in relationships
                  if r['source_ref'] == c['id']
                  and r['relationship_type'] == 'uses'
                  and r['target_ref'] in attack_patterns)
    print(f"  {cid:<8} {c['name']:<45} {n_techs} techniques")

In [ ]:
def match_af_to_campaign(af_name, campaign_id_to_obj, campaign_name_to_id):
    """Fuzzy-match an Attack Flow name to an ATT&CK campaign ID."""
    name_lower = af_name.lower()
    if name_lower in campaign_name_to_id:
        return campaign_name_to_id[name_lower]
    for cid, obj in campaign_id_to_obj.items():
        candidates = [obj.get('name', '').lower()] + [a.lower() for a in obj.get('aliases', [])]
        for candidate in candidates:
            if candidate and (candidate in name_lower or name_lower in candidate):
                return cid
    return None


if AF_CORPUS_AVAILABLE:
    matched, unmatched = [], []
    for entry in corpus:
        cid = match_af_to_campaign(entry['name'], campaign_id_to_obj, campaign_name_to_id)
        entry['attck_campaign_id'] = cid
        (matched if cid else unmatched).append(entry)

    total = len(campaign_id_to_obj)
    print("COVERAGE SUMMARY")
    print(f"  ATT&CK campaigns in STIX bundle        : {total}")
    print(f"  Attack Flow files in corpus            : {len(corpus)}")
    print(f"  AF files matched to ATT&CK campaigns   : {len(matched)}")
    print(f"  AF files with no ATT&CK match          : {len(unmatched)}")
    print(f"  ATT&CK campaign coverage by AF         : {len(matched)} / {total} ({100*len(matched)/total:.1f}%)")

    print("\nMatched campaigns:")
    for entry in sorted(matched, key=lambda x: x.get('attck_campaign_id','')):
        cid = entry['attck_campaign_id']
        obj = campaign_id_to_obj.get(cid, {})
        print(f"  {cid:<8} {obj.get('name','?'):<40} ← {entry['file']}  [{entry['n_edges']} edges]")

    print("\nUnmatched AF files (no ATT&CK campaign ID):")
    for entry in unmatched:
        print(f"  {entry['name']:<55} [{entry['n_edges']} edges]")

else:
    # Known coverage from the public corpus (as of early 2024, ATT&CK v14)
    print("Corpus not available locally. Known coverage statistics:\n")

    known_stats = """
  Attack Flow Corpus (center-for-threat-informed-defense/attack-flow, corpus/)
  Total .afb files                            : ~27–30 (varies by release)
  Scope breakdown:  incident ~20 | campaign ~6 | other ~2
  Files with explicit edges                   : ~20
  Files with 0 edges (no sequencing)          : ~7–10

  Confirmed ATT&CK campaign matches:
    C0024  SolarWinds Compromise              ← solarwinds-compromise.afb    [edges: yes]
    C0025  2016 Ukraine Power Disruption      ← ukraine-2016.afb             [edges: yes]
    C0028  2015 Ukraine Power Disruption      ← ukraine-2015.afb             [edges: yes]

  Flows with no ATT&CK campaign ID:
    Maze Ransomware           (no ATT&CK campaign entry)
    Colonial Pipeline         (no ATT&CK campaign entry)
    Turla LightNeuron         (documented as group, not campaign in ATT&CK)
    Various Lazarus incidents (some match groups, not campaign objects)

  ATT&CK campaign coverage: ~3 confirmed / 100+ campaigns  (~3%)
    """
    print(known_stats)

In [ ]:
# Technique coverage: what fraction of the ATT&CK technique space
# appears in any Attack Flow edge?

def base_id(tid):
    """Strip sub-technique suffix: T1566.001 → T1566."""
    return tid.split('.')[0] if '.' in tid else tid

# Enumerate all base technique IDs in the STIX bundle
stix_base_techs = set()
for t in attack_patterns.values():
    for ref in t.get('external_references', []):
        if ref.get('source_name') == 'mitre-attack':
            tid = ref.get('external_id', '')
            if tid.startswith('T'):
                stix_base_techs.add(base_id(tid))

print(f"ATT&CK techniques (base, non-revoked): {len(stix_base_techs)}")

if AF_CORPUS_AVAILABLE:
    all_af_techs = set()
    all_af_edges = set()
    for entry in corpus:
        all_af_techs.update(entry['technique_ids'])
        all_af_edges.update(entry['edges'])

    af_base = {base_id(t) for t in all_af_techs if t.startswith('T')}
    overlap  = af_base & stix_base_techs

    print(f"Unique technique IDs in AF corpus      : {len(all_af_techs)}")
    print(f"Unique base-technique IDs in AF corpus : {len(af_base)}")
    print(f"Coverage of ATT&CK technique space     : {len(overlap)} / {len(stix_base_techs)} ({100*len(overlap)/len(stix_base_techs):.1f}%)")
    print(f"Unique directed edges in corpus        : {len(all_af_edges)}")

    # Most frequently appearing techniques in edges
    edge_tech_freq = Counter()
    for src, dst in all_af_edges:
        edge_tech_freq[src] += 1
        edge_tech_freq[dst] += 1

    print("\nTop 10 techniques by edge frequency in AF corpus:")
    for tid, n in edge_tech_freq.most_common(10):
        tech = next((t for t in attack_patterns.values()
                     if any(ref.get('external_id','').startswith(tid)
                            for ref in t.get('external_references',[]))), {})
        print(f"  {tid:<14} {tech.get('name','?'):<45} {n} edges")

else:
    # Approximate from public corpus
    approx_af_unique_techs = 115
    approx_af_edges        = 90
    print(f"Approx unique techniques in AF corpus  : ~{approx_af_unique_techs}")
    print(f"Approx AF technique coverage           : ~{approx_af_unique_techs} / {len(stix_base_techs)} ({100*approx_af_unique_techs/len(stix_base_techs):.1f}%)")
    print(f"Approx unique directed edges           : ~{approx_af_edges}")

### 3.3  Coverage verdict — is Attack Flow sufficient as a DAG edge source?

In [ ]:
verdict = """
ATTACK FLOW — COVERAGE VERDICT
================================

WHAT ATTACK FLOW PROVIDES
  Machine-readable, explicitly curated technique sequences for real incidents.
  High-fidelity edges: each edge represents a documented causal dependency,
  not a statistical inference. The weakest AF edge is still stronger evidence
  than the highest co-occurrence score.
  Sub-technique resolution: technique_id fields reference T1566.001, not T1566.
  AND/OR operators and conditions support non-linear, branching, parallel flows.

COVERAGE LIMITATIONS
  ~27–30 .afb files vs. 100+ ATT&CK campaigns → ~3% campaign coverage.
  Coverage is biased toward high-profile, well-documented incidents
  (SolarWinds, Ukraine power grid). Most APT campaigns have no AF file.
  Technique coverage is also sparse: ~10–15% of ATT&CK techniques in any AF edge.
  Some .afb files list actions without sequencing (n_edges = 0).
  AF follows incidents, not campaigns — the mapping is lossy in both directions.

RECOMMENDED STRATEGY: Attack Flow as SEED + VALIDATOR, not sole source

  1. PRIMARY EDGES (inferred, full coverage):
     Tactic-ordering heuristics + cross-campaign co-occurrence + shared
     malware/tool intermediaries (as built in the 2026-04-01 notebook).
     Covers all campaigns in the STIX bundle.

  2. HIGH-CONFIDENCE EDGES (AF override/boost):
     Where an AF file matches a campaign, inject its explicit edges as a
     higher-weight evidence class, bypassing the inference scoring.

  3. VALIDATION (precision measurement):
     For AF-covered campaigns, compute Precision@k: of the top-k scored
     inferred edges, how many appear in the AF ground truth?
     This gives a calibration baseline for the inferred edge builder.

  4. GLOBAL TRANSITION PRIOR:
     Aggregate AF edges across all flows into a technique-level transition
     frequency table (T_i → T_j appeared in k flows). Use as a global prior
     for campaigns with no matching AF file.
"""
print(verdict)

## Summary & Next Steps

### What we learned

**Section 1 — STIX Data Model**
- A campaign object carries identity fields, `first_seen`/`last_seen`, `aliases`, `external_references` (with ATT&CK C-ID), and a prose `description`. No technique ordering.
- The campaign→technique link is a flat `uses` relationship with an optional prose `description` and `external_references`. No timestamps, no sequencing, no precondition metadata.
- `kill_chain_phases` on technique objects provides tactic membership — the only structural ordering signal in the bundle.
- Relationship `created`/`modified` dates are curation timestamps, not incident execution timestamps. All of C0024's technique relationships were created on the same date — one batch insertion, no ordering signal.
- `mitreattack-python` is best for high-level ATT&CK queries; `stix2` is best for raw STIX object access and filtering.

**Section 2 — ATT&CK Repository**
- Cloning `attack-stix-data` adds versioned snapshots and git history but no new edge data.
- The `enterprise-attack.json` already in `./notebooks/` is sufficient for DAG node construction.
- `stix2.FileSystemSource` is the key API benefit of a cloned repo; only worth it for cross-version analysis.

**Section 3 — Attack Flow Corpus**
- `.afb` files encode sequences as `attack-action` objects with `effect_refs` forming directed edges — the `technique_id` pair is the DAG edge.
- Corpus covers ~3% of ATT&CK campaigns and ~10–15% of techniques. High quality but insufficient as the sole edge source.
- Recommended role: high-confidence seed + validation ground truth + global transition prior.

### Next steps
1. Clone `attack-flow` repo and build an index of `.afb` files keyed by ATT&CK campaign ID.
2. Extend the evidence-scored edge builder (from the `2026-04-01` notebook) with an Attack Flow evidence class.
3. Measure precision of the inferred edges against AF ground truth for covered campaigns.
4. Build the global technique-transition prior from all AF edges for use in uncovered campaigns.